# CrewAI Teams

## Scenario: run a risk-aware EU checkout incident program

Conversion fell 35%. Observability, Release, Customer Impact, Incident Analyst, and Risk Reviewer roles prepare a structured proposal. **Safety boundary:** task ownership does not authorize production tools, customer communications, or a rollback. Those remain application-controlled and approval-gated.

![Multi-agent topology](../../../assets/multi-agent-patterns.svg)

CrewAI makes roles, tasks, deliverables, and context dependencies explicit. Use this shape when specialized artifacts improve the outcome; use a deterministic Flow/application controller for routing, state, approval, retries, and durable recovery.

## 1. Contract and task design

Specialist tasks are read-only and return attributed artifacts: metrics/log IDs, deployment timestamp/change, or customer segments/tickets. The analyst receives those artifacts as task context and must separate evidence from inference. The reviewer challenges unsupported claims, unsafe remediation, missing evidence, and customer-commitment risk. Terminals: accepted proposal, revision, escalation, or abstention.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'curriculum').exists(): ROOT=next(p for p in (ROOT,*ROOT.parents) if (p/'curriculum').exists())
sys.path.insert(0,str(ROOT/'curriculum'/'advanced'/'05-incident-response-capstone'))
from agentops_lab.crewai_team import run_crewai_shaped_team
result=run_crewai_shaped_team()
for event in result['trace']:
    print(event['task'],'|',event['agent_role'],'| context=',event['uses_context'])
    print(' ',event['output'][:160])
print(result['comparison'])

metrics_task | Observability Engineer | context= none
  eu-west checkout-to-payment conversion is down 35%; cart-to-checkout clicks and payment authorization errors are normal; 3DS callback errors increased.
deployment_task | Release Engineer | context= none
  checkout-ui 2026-08-07.1 changed VAT validation and 3DS redirect handling for eu-west before the conversion drop.
customer_task | Customer Impact Analyst | context= none
  Enterprise VAT-registered EU customers are disproportionately affected; support tickets describe a 3DS redirect loop after VAT entry.
analysis_task | Incident Commander | context= metrics_task, deployment_task, customer_task
  Likely cause: eu-west checkout UI VAT/3DS redirect change. Recommended plan: disable the feature flag or roll back checkout-ui in eu-west, monitor checkout-to-p
{'crewai_help': 'Clear role/task mapping and readable collaboration plan.', 'langgraph_control': 'More explicit state, branching, persistence, and policy checkpoints.', 'autogen_c

## 2. CrewAI feature mapping

`Agent` models role/goal/tools; `Task` models a typed deliverable and its context; `Crew` assembles the plan. Sequential processing fits causal task dependencies. Hierarchical processing can delegate, but a manager needs scoped delegation, budgets, and escalation. Flows are preferable for deterministic routing, approval, persistence, retries, and lifecycle state. Optional callbacks/guardrails should validate schema and evidence but cannot replace resource-side policy.

In [2]:
# Deliberate failure experiment: a reviewer must not accept a claim without sufficient artifacts.
artifacts={'telemetry':'3DS callback errors','deployment':None,'impact':'EU VAT tickets'}
missing=[name for name,value in artifacts.items() if not value]
decision='escalate-for-evidence' if missing else 'review-proposal'
print('missing:',missing,'decision:',decision)
assert decision=='escalate-for-evidence'

# Simple work should bypass coordination.
simple_task={'known_steps':True,'cross_domain_evidence':False}
route='deterministic-workflow' if simple_task['known_steps'] else 'crew'
assert route=='deterministic-workflow'

missing: ['deployment'] decision: escalate-for-evidence


## 3. Evaluation and production readiness

Compare a crew with a bounded single-agent baseline: outcome/evidence correctness, risk-review catch rate, unsupported proposal rate, tool calls, latency, cost, task duplication, and escalation quality. Record task owner, context artifacts, tool calls, outputs, validation, cost, and termination. Apply tenant/identity/tool/approval/idempotency/budget controls around—not inside—the crew.

**Exercises:** add a compliance-review task; make a hierarchical manager and constrain delegation; run parallel read-only specialists with a concurrency cap; design a typed incident brief; and set an evidence/cost/latency release gate.

References: [CrewAI Tasks](https://docs.crewai.com/en/concepts/tasks), [Crews](https://docs.crewai.com/en/concepts/crews), [Flows](https://docs.crewai.com/en/concepts/flows).